In [3]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder

data = pd.read_csv("laptop.csv")
data.head()

In [7]:
data['Price'] = data['Price'].replace('[^0-9]', '', regex=True)
data['Price'] = data['Price'].astype(int)

In [8]:
def budget(price):
    if price < 50000:
        return "Low"
    elif price < 100000:
        return "Medium"
    else:
        return "High"

data['Budget'] = data['Price'].apply(budget)

In [9]:
print(data.columns)

Index(['Unnamed: 0', 'Model', 'Price', 'Rating', 'Generation', 'Core', 'Ram',
       'SSD', 'Display', 'Graphics', 'OS', 'Warranty', 'Budget'],
      dtype='object')


In [10]:
data = data.drop(columns=['Unnamed: 0'])

In [11]:
print(data.columns)

Index(['Model', 'Price', 'Rating', 'Generation', 'Core', 'Ram', 'SSD',
       'Display', 'Graphics', 'OS', 'Warranty', 'Budget'],
      dtype='object')


In [12]:
data['Budget'] = data['Price'].apply(budget)
data.head()

,Model,Price,Rating,Generation,Core,Ram,SSD,Display,Graphics,OS,Warranty,Budget
0,HP Victus 15-fb0157AX Gaming Laptop (AMD Ryzen...,50399,70.0,5th Gen AMD Ryzen 5 5600H,"Hexa Core, 12 Threads",8 GB DDR4 RAM,512 GB SSD,"15.6 inches, 1920 x 1080 pixels, Touch Screen",4 GB AMD Radeon RX 6500M,Windows 11 OS,1 Year Warranty,Medium
1,Tecno Megabook T1 Laptop (11th Gen Core i3/ 8G...,23990,56.0,11th Gen Intel Core i3 1115G4,"Dual Core, 4 Threads",8 GB LPDDR4 RAM,512 GB SSD,"15.6 inches, 1920 x 1080 pixels",Intel UHD Graphics,Windows 11 OS,1 Year Warranty,Low
2,Lenovo V15 G4 ‎82YU00W7IN Laptop (AMD Ryzen 3 ...,26989,55.0,7th Gen AMD Ryzen 3 7320U,"Quad Core, 8 Threads",8 GB LPDDR5 RAM,512 GB SSD,"15.6 inches, 1920 x 1080 pixels",AMD Radeon Graphics,Windows 11 OS,1 Year Warranty,Low
3,Samsung Galaxy Book2 Pro 13 Laptop (12th Gen C...,69990,60.0,12th Gen Intel Core i5 1240P,"12 Cores (4P + 8E), 16 Threads",16 GB LPDDR5 RAM,512 GB SSD,"13.3 inches, 1080 x 1920 pixels",Intel Iris Xe Graphics,Windows 11 OS,1 Year Warranty,Medium
4,Xiaomi Redmi G Pro 2024 Gaming Laptop (14th Ge...,102990,78.0,14th Gen Intel Core i9 14900HX,"24 Cores (8P + 16E), 32 Threads",16 GB DDR5 RAM,1 TB SSD,"16.1 inches, 2560 x 1600 pixels",8 GB NVIDIA GeForce RTX 4060,Windows 11 OS,1 Year Warranty,High


In [15]:
def usage(row):
    # We use 'Graphics' because that is the name in your dataset
    if "RTX" in str(row['Graphics']) or "GTX" in str(row['Graphics']):
        return "Gaming"
    # We use 'Ram' because it's capitalized that way in your columns
    elif "16" in str(row['Ram']):
        return "Programming"
    else:
        return "Study"

data['Usage'] = data.apply(usage, axis=1)

In [16]:
from sklearn.preprocessing import LabelEncoder

le_budget = LabelEncoder()
le_usage = LabelEncoder()

data['Budget_num'] = le_budget.fit_transform(data['Budget'])
data['Usage_num'] = le_usage.fit_transform(data['Usage'])

In [23]:
from sklearn.preprocessing import LabelEncoder

le_budget = LabelEncoder()
le_usage = LabelEncoder()
le_model = LabelEncoder() # New encoder for laptop names

data['Budget_num'] = le_budget.fit_transform(data['Budget'])
data['Usage_num'] = le_usage.fit_transform(data['Usage'])
data['Model_num'] = le_model.fit_transform(data['Model']) # Encoding the Model column

In [26]:
from sklearn.preprocessing import LabelEncoder

# 1. Initialize the encoders
le_budget = LabelEncoder()
le_usage = LabelEncoder()
le_model = LabelEncoder()
le_ram = LabelEncoder()
le_ssd = LabelEncoder()
le_gpu = LabelEncoder()

# 2. Create the new columns in your 'data' dataframe
# We are creating these names right now so Step 8 can find them!
data['Budget_num']   = le_budget.fit_transform(data['Budget'])
data['Usage_num']    = le_usage.fit_transform(data['Usage'])
data['Model_num']    = le_model.fit_transform(data['Model'])
data['Ram_num']      = le_ram.fit_transform(data['Ram'])
data['SSD_num']      = le_ssd.fit_transform(data['SSD'])
data['Graphics_num'] = le_gpu.fit_transform(data['Graphics'])

# 3. VERIFY - Let's make sure they are there
print("New columns created:")
print(data[['Budget_num', 'Usage_num', 'Model_num', 'Ram_num', 'SSD_num', 'Graphics_num']].head())

New columns created:
   Budget_num  Usage_num  Model_num  Ram_num  SSD_num  Graphics_num
0           2          2        488       47       11            28
1           1          2        893       50       11           134
2           1          2        749       54       11            84
3           2          1        891       15       11           128
4           0          0        909        8        6            67


In [27]:
from sklearn.tree import DecisionTreeClassifier

# Inputs
X = data[['Budget_num', 'Usage_num']]

# Outputs (Now these columns definitely exist!)
y = data[['Model_num', 'Ram_num', 'SSD_num', 'Graphics_num']]

model = DecisionTreeClassifier()
model.fit(X, y)

print("✅ Model trained successfully!")

✅ Model trained successfully!


In [33]:
import time
import os
from IPython.display import clear_output # Clears the Jupyter junk

# --- THE UI BLOCK ---
print("="*50)
print(" 🤖  AI LAPTOP ADVISOR v1.0  🤖 ".center(50))
print("="*50)

# 1. Get Inputs
budget_in = input("💰 Enter Budget (Low/Medium/High): ").strip().capitalize()
usage_in = input("🎮 Enter Usage (Study/Programming/Gaming): ").strip().capitalize()

try:
    # 2. Process
    b_num = le_budget.transform([budget_in])[0]
    u_num = le_usage.transform([usage_in])[0]

    # Clear the screen so only the result shows
    clear_output(wait=True) 
    
    print("="*50)
    print(" 🔎  SCANNING DATABASE FOR YOUR MATCH...  🔎 ".center(50))
    print("="*50)
    time.sleep(1)

    # 3. Predict
    import pandas as pd
    test_data = pd.DataFrame([[b_num, u_num]], columns=['Budget_num', 'Usage_num'])
    prediction = model.predict(test_data)

    # 4. Decode
    final_model = le_model.inverse_transform([int(prediction[0][0])])[0]
    final_ram   = le_ram.inverse_transform([int(prediction[0][1])])[0]
    final_ssd   = le_ssd.inverse_transform([int(prediction[0][2])])[0]
    final_gpu   = le_gpu.inverse_transform([int(prediction[0][3])])[0]

    # 5. THE CLEAN OUTPUT
    print("\n" + "✅ MATCH FOUND!".center(50))
    print("╔" + "═"*48 + "╗")
    print(f"║  🏆 MODEL:    {final_model.ljust(33)} ║")
    print(f"║  ────────────────────────────────────────────  ║")
    print(f"║  🧠 RAM:      {final_ram.ljust(33)} ║")
    print(f"║  💾 STORAGE:  {final_ssd.ljust(33)} ║")
    print(f"║  🖥️  GRAPHICS: {final_gpu.ljust(33)} ║")
    print("╚" + "═"*48 + "╝")
    print(f"\nConfiguration optimized for {usage_in} ({budget_in} Budget).".center(50))
    print("="*50)

except ValueError:
    print("\n❌ INPUT ERROR: Please type the words exactly.")
    print(f"Choices: {', '.join(le_budget.classes_)} / {', '.join(le_usage.classes_)}")

    🔎  SCANNING DATABASE FOR YOUR MATCH...  🔎     

                  ✅ MATCH FOUND!                  
╔════════════════════════════════════════════════╗
║  🏆 MODEL:    AXL Vayu Book LAP01 Laptop (Celeron N4020/ 4GB/ 128GB SSD / Win11 Home) ║
║  ────────────────────────────────────────────  ║
║  🧠 RAM:      8 GB DDR4 RAM                     ║
║  💾 STORAGE:  512 GB SSD                        ║
║  🖥️  GRAPHICS: Intel UHD Graphics                ║
╚════════════════════════════════════════════════╝
 
Configuration optimized for Study (Low Budget). 
